# Single-task conditioning overfitting ladder on Modal

This diagnostic asks whether a fresh full FLUX transformer can first memorize one fixed ARC target, then route two contradictory NVARC conditions, and finally overfit eight repeated conditions. NVARC and the FLUX VAE remain frozen; every stage uses the padded 30x30 canvas and starts independently from the same FLUX basis checkpoint.

The stages are:

1. `one-condition`: canonical repeated eight times per batch for 256 updates;
2. `two-condition-routing`: canonical and one strongly color-permuted ID view, four times each, for 512 updates;
3. `eight-condition-overfit`: eight fixed geometry/color views, once each per batch, for 512 updates.

Each optimizer batch receives one jittered timestep from every eighth of `[0, 1)` and independent Gaussian noise per row. Original pair 4 is always the target; pairs 0--3 are always the demonstrations. Stage 1 is only a mechanics control. Exact routing in stage 2 is the first direct evidence that FLUX uses NVARC conditioning.

Prerequisites:

- one Modal H200;
- the writable `diffusion-arc-models` Volume mounted at `/mnt/diffusion-arc-models`;
- the model checkpoints and complete ARC-AGI-2 data already present in that Volume;
- a `GITHUB_TOKEN` Modal Secret;
- `DIFFUSION_ARC_REVISION` set to the exact 40-character pushed commit containing this experiment.

Run the notebook strictly from top to bottom. The long runner cell performs all three independent stages, streams progress, uploads three checkpoints, and verifies the final report byte-for-byte through the Modal Volume API.


In [ ]:
!nvidia-smi


In [ ]:
%uv pip uninstall torchvision


## 1. Immutable run configuration

Set `DIFFUSION_ARC_REVISION` to the exact pushed commit before running this cell. A branch name such as `main` is intentionally rejected so the report always identifies immutable source.


In [ ]:
from __future__ import annotations

import base64
from datetime import datetime, timezone
import importlib
import importlib.metadata
import json
import os
from pathlib import Path
import subprocess
import sys
import time

VOLUME_NAME = 'diffusion-arc-models'
VOLUME_ROOT = Path('/mnt/diffusion-arc-models')
REPOSITORY_URL = 'https://github.com/joshuakauling/diffusion-arc.git'
SOURCE_REVISION = "202a10316bc48d18e7d6e98755de87d118b5a032"
if len(SOURCE_REVISION) != 40 or any(character not in '0123456789abcdefABCDEF' for character in SOURCE_REVISION):
    raise ValueError('DIFFUSION_ARC_REVISION must be an exact 40-character Git commit')
AUTO_SHUTDOWN = os.environ.get('DIFFUSION_ARC_AUTO_SHUTDOWN', '1') == '1'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
RESULT_FAMILY = 'single-task-conditioning-overfitting-ladder'
RESULTS_ROOT = VOLUME_ROOT / 'results' / RESULT_FAMILY
REPORT_PATH = RESULTS_ROOT / RUN_ID / 'report.json'
EXPECTED_STAGES = (
    'one-condition',
    'two-condition-routing',
    'eight-condition-overfit',
)

if not VOLUME_ROOT.is_dir():
    raise FileNotFoundError(f'Modal Volume mount is missing: {VOLUME_ROOT}')
if REPORT_PATH.exists():
    raise FileExistsError(REPORT_PATH)
print('Run ID:', RUN_ID)
print('Requested source revision:', SOURCE_REVISION)
print('Final report:', REPORT_PATH)


## 2. Fetch, install, and test the exact source

This cell fetches only the requested commit, installs mismatched pinned packages, installs Diffusion-ARC editable from that checkout, and requires the complete unit-test suite to pass before GPU training begins.


In [ ]:
def run_command(label: str, command: list[str], *, cwd: Path | None = None, env=None) -> str:
    completed = subprocess.run(
        command, cwd=cwd, env=env, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, check=False,
    )
    if completed.returncode != 0:
        tail = '\n'.join(completed.stdout.splitlines()[-50:])
        if tail:
            print(tail)
        raise RuntimeError(f'{label} failed with exit code {completed.returncode}')
    return completed.stdout.strip()

SOURCE_DIR = Path('/tmp') / f'diffusion-arc-conditioning-ladder-source-{RUN_ID}'
SOURCE_DIR.mkdir(parents=True)
run_command('git init', ['git', 'init'], cwd=SOURCE_DIR)
run_command('add origin', ['git', 'remote', 'add', 'origin', REPOSITORY_URL], cwd=SOURCE_DIR)

github_token = os.environ.get('GITHUB_TOKEN', '').strip()
if not github_token:
    raise RuntimeError('Attach a Modal Secret containing GITHUB_TOKEN')
credential = base64.b64encode(
    f'x-access-token:{github_token}'.encode('utf-8')
).decode('ascii')
git_environment = os.environ.copy()
git_environment.pop('GITHUB_TOKEN', None)
git_environment.update({
    'GIT_TERMINAL_PROMPT': '0',
    'GIT_CONFIG_COUNT': '1',
    'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
    'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {credential}',
})
try:
    run_command(
        'fetch source',
        ['git', 'fetch', '--depth=1', 'origin', SOURCE_REVISION],
        cwd=SOURCE_DIR, env=git_environment,
    )
    run_command('checkout source', ['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=SOURCE_DIR)
finally:
    git_environment.clear()
    del credential
RESOLVED_SOURCE_COMMIT = run_command(
    'resolve source', ['git', 'rev-parse', 'HEAD'], cwd=SOURCE_DIR
)
if RESOLVED_SOURCE_COMMIT.lower() != SOURCE_REVISION.lower():
    raise RuntimeError('Fetched source commit differs from requested revision')

PINNED_PACKAGES = {
    'accelerate': '1.12.0', 'diffusers': '0.39.0', 'einops': '0.8.1',
    'Pillow': '11.3.0', 'safetensors': '0.8.0', 'torch': '2.13.0',
    'tqdm': '4.67.1', 'transformers': '4.56.1',
}
def installed_version(name: str) -> str | None:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None
requirements = [
    f'{name}=={version}' for name, version in PINNED_PACKAGES.items()
    if installed_version(name) != version
]
if requirements:
    run_command(
        'install pinned dependencies',
        [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-input', '--no-cache-dir', *requirements],
    )
run_command(
    'install Diffusion-ARC source',
    [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-input',
     '--no-deps', '--no-build-isolation', '-e', str(SOURCE_DIR)],
)
source_path = str((SOURCE_DIR / 'src').resolve())
if source_path not in sys.path:
    sys.path.insert(0, source_path)
importlib.invalidate_caches()
test_environment = os.environ.copy()
test_environment['PYTHONDONTWRITEBYTECODE'] = '1'
test_environment['PYTHONPATH'] = source_path
test_started = time.perf_counter()
test_process = subprocess.run(
    [sys.executable, '-B', '-m', 'unittest', 'discover', '-s', 'tests'],
    cwd=SOURCE_DIR, env=test_environment, stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT, text=True, check=False,
)
print('\n'.join(test_process.stdout.splitlines()[-30:]))
if test_process.returncode != 0:
    raise RuntimeError(f'Unit tests returned {test_process.returncode}')
print('Resolved source commit:', RESOLVED_SOURCE_COMMIT)
print(f'Unit tests: PASS in {time.perf_counter() - test_started:.1f}s')


## 3. Run the three independent stages

The runner validates the raw task and all 17 conditions, caches one physical NVARC batch and one VAE batch, executes the three fresh-from-base FLUX stages, evaluates fixed flow states and full generation, and uploads one checkpoint per stage. Progress bars show update rate, loss, gradient norm, and peak GPU memory.


In [ ]:
runner = SOURCE_DIR / 'scripts' / 'experiments' / 'run_single_task_conditioning_ladder.py'
if not runner.is_file():
    raise RuntimeError(
        'Selected source revision lacks the conditioning-ladder runner; '
        'push the implementation and select that exact commit.'
    )
runner_command = [
    sys.executable, '-u', str(runner),
    '--volume-root', str(VOLUME_ROOT),
    '--volume-name', VOLUME_NAME,
    '--run-id', RUN_ID,
    '--source-commit', RESOLVED_SOURCE_COMMIT,
]
runner_environment = os.environ.copy()
runner_environment['PYTHONPATH'] = source_path
process = subprocess.Popen(
    runner_command, cwd=SOURCE_DIR, env=runner_environment,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    bufsize=1,
)
assert process.stdout is not None
for output_line in process.stdout:
    print(output_line, end='')
return_code = process.wait()
if return_code != 0:
    if REPORT_PATH.is_file():
        failed_report = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
        print('Failed stage:', failed_report.get('failed_stage'))
        print('Failures:', failed_report.get('failures'))
    raise RuntimeError(f'Conditioning-ladder runner exited with {return_code}')
REPORT = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
if REPORT.get('status') != 'COMPLETE':
    raise RuntimeError(f'Experiment report is not COMPLETE: {REPORT.get("status")}')
if not REPORT.get('durability', {}).get('remote_content_verified'):
    raise RuntimeError('Final report was not verified after batch upload')
REPORT['comparison']


In [ ]:
failed_report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))

print(json.dumps({
    "report_path": str(REPORT_PATH),
    "status": failed_report.get("status"),
    "failed_stage": failed_report.get("failed_stage"),
    "failures": failed_report.get("failures"),
}, indent=2))

## 4. Checkpoint and result locations


In [ ]:
checkpoint_summary = {}
for stage_name in EXPECTED_STAGES:
    stage = REPORT['stages'][stage_name]
    checkpoint = stage.get('saved_checkpoint')
    if stage.get('status') != 'COMPLETE':
        raise RuntimeError(f'{stage_name} is not COMPLETE')
    if not isinstance(checkpoint, dict) or not checkpoint.get('remote_file_sizes_verified'):
        raise RuntimeError(f'{stage_name} checkpoint is not remotely verified')
    checkpoint_summary[stage_name] = {
        'checkpoint': checkpoint['path'],
        'step': checkpoint['step'],
        'state_scope': checkpoint['state_scope'],
        'target_exposures': stage['training']['target_exposures'],
    }
{
    'report': str(REPORT_PATH),
    'remote_report': REPORT['durability']['remote_report_path'],
    'comparison': REPORT['comparison'],
    'checkpoints': checkpoint_summary,
}


## 5. Optional automatic shutdown

Shutdown is scheduled only after all three checkpoint directories have matching remote file sizes and the final report has been uploaded and read back byte-for-byte. Set `DIFFUSION_ARC_AUTO_SHUTDOWN=0` before starting to keep the kernel alive.


In [ ]:
durable_report = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
if durable_report.get('status') != 'COMPLETE':
    raise RuntimeError('Refusing shutdown: report is not COMPLETE')
if not durable_report.get('durability', {}).get('remote_content_verified'):
    raise RuntimeError('Refusing shutdown: remote report content is unverified')
for stage_name in EXPECTED_STAGES:
    checkpoint = durable_report['stages'][stage_name].get('saved_checkpoint')
    if not isinstance(checkpoint, dict) or not checkpoint.get('remote_file_sizes_verified'):
        raise RuntimeError(f'Refusing shutdown: {stage_name} checkpoint is unverified')
if not AUTO_SHUTDOWN:
    print('Automatic shutdown disabled; report and all three checkpoints are durable.')
else:
    modal_task_id = os.environ.get('MODAL_TASK_ID', '').strip()
    if not modal_task_id:
        raise RuntimeError('MODAL_TASK_ID is unavailable; stop the kernel manually')
    shutdown_code = (
        'import subprocess, sys, time; ' 'time.sleep(10); '
        "subprocess.run(['modal', 'container', 'stop', '--yes', sys.argv[1]], check=False)"
    )
    subprocess.Popen(
        [sys.executable, '-c', shutdown_code, modal_task_id],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True,
    )
    print('All artifacts verified. Modal shutdown scheduled in 10 seconds.')


## Interpretation after the run

- Stage 1 exact success only proves that the padded flow path can memorize one target.
- Stage 1 success with stage 2 failure points to the NVARC-to-FLUX conditioning path.
- Stage 2 exact success with a clear correct-versus-swapped gap is the minimal evidence that direct NVARC conditioning can route contradictory outputs.
- Stage 3 seen success is the requested one-task conditional-overfitting result; unseen success is additional presentation-transfer evidence.
- `COMPLETE` means the technical protocol and durable uploads completed. It does not mean that any learning criterion passed.
